# ***텍스트 데이터의 처리 응용-2***
## ***오픈소스 거대언어모델(LLM)과 프로그래밍 언어 결합하기***

## ***오픈소스 Tokenizer 다운로드***

In [2]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",bos_token='</s>', eos_token='</s>', unk_token='<unk>',pad_token='<pad>', mask_token='<mask>')
tokenizer.save_pretrained("./my_tokenizer")

tokenizer.json: 0.00B [00:00, ?B/s]

('./my_tokenizer/tokenizer_config.json', './my_tokenizer/tokenizer.json')

### ***Tokenizer를 이용해 문장을 숫자로 변환하기***
- 지난 시간에 우리가 직접 만든 Tokenizer와 결과를 비교해봅니다

In [4]:
# 입력 문장
text = "주말 미세먼지 재난 문자 속출 야외 활동 자제 요청"

# 토큰화 및 ID 변환
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(tokens)
print(token_ids)

['▁주', '말', '▁미세', '먼', '지', '▁재난', '▁문자', '▁속', '출', '▁야외', '▁활동', '▁자제', '▁요청']
[9046, 7492, 27073, 7514, 8263, 31638, 16378, 9238, 8400, 20456, 9478, 22425, 11544]


### ***vocab 들여다보기***
- 지난 시간에 우리가 직접 만든 Tokenizer와 어떤 차이가 있는지 확인해봅니다

In [ ]:
import json

with open('./my_tokenizer/tokenizer.json', 'r') as f:
    j = json.load(f)

j

{'version': '1.0',
 'truncation': None,
 'padding': None,
 'added_tokens': [{'id': 0,
   'content': '<s>',
   'single_word': False,
   'lstrip': False,
   'rstrip': False,
   'normalized': False,
   'special': True},
  {'id': 1,
   'content': '</s>',
   'single_word': False,
   'lstrip': False,
   'rstrip': False,
   'normalized': False,
   'special': True},
  {'id': 2,
   'content': '<usr>',
   'single_word': False,
   'lstrip': False,
   'rstrip': False,
   'normalized': False,
   'special': True},
  {'id': 3,
   'content': '<pad>',
   'single_word': False,
   'lstrip': False,
   'rstrip': False,
   'normalized': False,
   'special': True},
  {'id': 4,
   'content': '<sys>',
   'single_word': False,
   'lstrip': False,
   'rstrip': False,
   'normalized': False,
   'special': True},
  {'id': 5,
   'content': '<unk>',
   'single_word': False,
   'lstrip': False,
   'rstrip': False,
   'normalized': False,
   'special': True},
  {'id': 6,
   'content': '<mask>',
   'single_word': False

### ***ollama 프레임워크 설치하기***
- 오픈소스 LLM을 다운로드 받고 사용하기 쉽게 도와주는 프레임워크 입니다

In [ ]:
! curl -fsSL https://ollama.com/install.sh | sh
#
# 그리고 Google colab의 좌측 하단의 터미널을 열고 아래와 같이 명령을 입력합니다
# ollama serve

### ***ollama python 라이브러리 설치***
- pip 명령을 이용해서 python 라이브러리를 설치합니다
- 우리는 ollama라는 python 라이브러리를 이용해서 우리의 프로그램과 LLM을 연결할 것입니다

In [ ]:
! pip install ollama

### ***언어모델 다운로드 - 1***
- gemma 이라는 언어모델을 다운로드 합니다
- 모델의 규모는 1b 입니다
- 10억개의 파라미터를 가진 모델 입니다

In [ ]:
! ollama pull gemma3:1b

### ***언어모델 다운로드 - 2***
- llama3.2 라는 언어모델을 다운로드 합니다
- 모델의 규모는 1b 입니다
- 10억개의 파라미터를 가진 모델 입니다

In [ ]:
! ollama pull llama3.2:1b

## ***각 모델 테스트***

- ollama 패키지의 chat 함수를 이용해서 LLM과 통신합니다
- LLM의 응답에서 생성된 문장 내용을 추출하여 출력합니다
- llama3.2:1b와 과 qwen:0.5b 모두 동일하게 실행해봅니다

In [ ]:
import ollama

model='llama3.2:1b'
# model='gemma3:1b'

messages = [{
        'role': 'user',
        'content': "Tell me this situation as a simple one word, 'I got A+ in my Math class!'"
    }]

response = ollama.chat(
    model=model,
    messages=messages,
    options={
        'temperature': 0.8
    })
content = response['message']['content']

print(content)

In [ ]:
import ollama

model='gemma3:1b'

messages = [{
        'role': 'user',
        'content': "Tell me this situation as a simple one word, 'I got A+ in my Math class!'"
    }]

response = ollama.chat(
    model=model,
    messages=messages,
    options={
        'temperature': 0.8
    })
content = response['message']['content']

print(content)

## ***세 언어모델간의 대화***
- llama3.2:1b, gemma3:1b, qwen3:0.6b 모델이 서로 10번 대화를 주고 받습니다

In [ ]:
import ollama

# 아래와 같이 2개의 모델을 사용합니다
model_1 = "gemma3:1b"
model_2 = "llama3.2:1b"
model_3 = "qwen3:0.6b"

# model1 에게 먼저 대화를 시작합니다
message_from_model_1 = ""
message_from_model_me = "Now, let's have an argument with anything. Any topic OK but one sentence per one turn. If you or I tell 'I lost' then I or you win"

print(f"Me: {message_from_model_me}")
response_3 = ollama.chat(
    model=model_3,
    messages=[{"role": "user", "content": message_from_model_me}],
    options={
        'temperature': 0.8
    })
message_from_model_3 = response_3['message']['content']


# 대화를 10번 반복합니다
for i in range(10):
    # model1의 응답을 받아 모델2가 말하고 이를 모델1에게 전달합니다
    print(f"Model 3 ({model_3}): {message_from_model_3}")
    response_2 = ollama.chat(
        model=model_2,
        messages=[{"role": "user", "content": message_from_model_3}],
        options={
            'temperature': 0.8
        })
    message_from_model_2 = response_2['message']['content']  # model2의 응답을 받습니다

    print("\n===============================")

    # model2의 응답을 받아 모델2가 말하고 이를 모델1에게 전달합니다
    print(f"Model 2 ({model_2}): {message_from_model_2}")
    response_1 = ollama.chat(
        model=model_1,
        messages=[{"role": "user", "content": message_from_model_2}],
        options={
            'temperature': 0.8
        })
    message_from_model_1 = response_1['message']['content']  # model1의 응답을 받습니다

    print(f"Model 1 ({model_1}): {message_from_model_1}")

    response_3 = ollama.chat(
        model=model_3,
        messages=[{"role": "user", "content": message_from_model_1}],
        options={
            'temperature': 0.8
        })
    message_from_model_3 = response_3['message']['content']

## ***한글로 시도***
- 시작 언어를 한글로 바꿔서 다시 시도해봅니다

In [ ]:
import ollama

# 아래와 같이 2개의 모델을 사용합니다
model_1 = "gemma3:1b"
model_2 = "llama3.2:1b"

# model1 에게 먼저 대화를 시작합니다
message_from_model_1 = ""
message_from_me = "오늘 점심 뭐먹지?"

# 초기 메시지를 model1에게 전달합니다
print(f"Me: {message_from_me}")
response_1 = ollama.chat(
    model=model_1,
    messages=[{"role": "user", "content": message_from_me}],
    options={
        'temperature': 0.8
    })
message_from_model_1 = response_1['message']['content']  # model1의 응답을 받습니다

# 대화를 10번 반복합니다
for i in range(10):
    # model1의 응답을 받아 모델2가 말하고 이를 모델1에게 전달합니다
    print(f"Model 1 ({model_1}): {message_from_model_1}")
    response_2 = ollama.chat(
        model=model_2,
        messages=[{"role": "user", "content": message_from_model_1}],
        options={
            'temperature': 0.8
        })
    message_from_model_2 = response_2['message']['content']  # model2의 응답을 받습니다

    print("\n===============================")

    # model2의 응답을 받아 모델2가 말하고 이를 모델1에게 전달합니다
    print(f"Model 2 ({model_2}): {message_from_model_2}")
    response_1 = ollama.chat(
        model=model_1,
        messages=[{"role": "user", "content": message_from_model_2}],
        options={
            'temperature': 0.8
        })
    message_from_model_1 = response_1['message']['content']  # model1의 응답을 받습니다